# Energy criticality

This notebook checks the results of the N-1 criticality analysis of the `inferred-provided-<region>` network, which records, for each substation, transmission line and power plant, the unmet demand when that component alone fails. The rule `build_energy_criticality` (`workflow/2-losses/energy.smk`, implemented in `src/energy/criticality.py`) writes the results; the notebook reads and plots them and builds nothing.

## Method

The analysis sets the demand of each substation, models the transmission network and runs one power flow for each failed component.

1. **Demand by substation area.** Each substation supplies its service area, the part of the island closer to it than to any other substation. An area's share of its island's demand is 0.6 times its share of the island's night-light radiance plus 0.4 times its share of the population: PyPSA-Earth's weighting of GDP and population, with night lights in place of GDP. The share multiplies the island's peak or average demand from the utility's annual report, and the result is placed on the substation's bus. An island without provided transmission lines is modelled as a single bus.
2. **Transmission network.** The provided transmission lines connect the substations, and each plant is attached to its substation. A line's rating is its number of circuits times the thermal rating of one circuit of a standard PyPSA line type, chosen by the line's design voltage.
3. **Failure of each component.** Starting each time from the intact network, the analysis removes one component: a substation together with its plants and its area's demand, a transmission line with all its segments, or a single plant. A linear optimal power flow (PyPSA) then supplies as much demand as the remaining network allows; the demand it cannot supply is the unmet demand, demand minus supply.
4. **Causes of unmet demand.** Demand is unmet when (a) its own substation fails; (b) its part of the network is islanded without synchronous generation (the `island_carriers` printed below), since inverter-based solar and wind farms disconnect when they lose the grid; (c) the generation left in its part of the network cannot cover the demand; or (d) the remaining lines would exceed their ratings.

Each failure is solved at peak and at average demand, which gives two quantities: `unmet_peak_mw`, the unmet demand in the worst hour, and `unmet_mwh_per_day`, the unmet energy per day of outage, which is the unmet demand at average demand over 24 hours. A restoration time in days turns the second into the energy not supplied. It is exact when whole areas lose supply, and too low for shortfalls that occur only near peak, as when a plant fails on a small island.

In [ ]:
import importlib
import json
import pathlib
import sys

for _candidate in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_candidate / "_helpers.py").exists():
        sys.path.insert(0, str(_candidate))
        break
# _helpers finds the workflow's files and draws them.
import _helpers as h  # noqa: E402

importlib.reload(h)  # so edits to _helpers.py apply without restarting the kernel
if not h.found(h.CRITICALITY_DIR / "criticality.csv", h.RUN_CRITICALITY):
    raise RuntimeError("Make the criticality tables first, with the command above.")
summary, by_area = h.load_criticality()
metadata = json.loads((h.CRITICALITY_DIR / "metadata.json").read_text())
print("model data:                    ", h.MODEL_DATA)
print("island_carriers:               ", metadata["island_carriers"])
print("unmet demand, intact network:   ", metadata["unmet_with_nothing_failed"])

## Demand by substation area

The first table gives each island's peak and average demand and its source (`demand_levels.csv`); the second gives each area's share of its island's demand (`service_weights_substations.csv`) and the resulting demand in MW. The rule `build_energy_demand` writes both files to `data/processed/energy/<model_data>/demand/inferred-provided-<region>/`. A substation's failure leaves the whole demand of its area unmet.

In [ ]:
display(h.demand_levels())
h.area_demand(summary).round({"share_of_island": 3, "peak_mw": 1, "average_mw": 1})

## Transmission line ratings

The map shows the rating of each transmission line, from its design voltage (colour) and number of circuits (width) in the line circuits table of the incoming data. Every line runs at the network's transmission voltage (`inferred.transmission_voltage_kv` in the energy config); a line designed for a higher voltage is given a larger standard conductor and so has a higher rating at that voltage.

In [ ]:
# clip: None for the islands with transmission lines, or a (minx, miny, maxx, maxy) box in degrees
h.plot_line_ratings(clip=None);

## Unmet demand when each component fails, by cause

The table lists every component whose failure leaves demand unmet, with its two quantities, and splits the unmet demand at peak by the causes in the method: `own_area_peak_mw` is the demand of the failed substation's own area (a); `cut_off` and `cut_off_peak_mw` are the other areas that lose all their supply, as when they are islanded without synchronous generation (b); and `partly_supplied_peak_mw` is the unmet demand in areas that keep some supply (c or d). `line_limits_peak_mw` is the part due to line ratings (d), and shows whether any single failure overloads the remaining lines. Components not listed leave no demand unmet.

In [ ]:
unmet = h.unmet_by_cause(summary, by_area)
unmet.round(1)

## Criticality of transmission lines and substations

The map colours each transmission line and substation by the unmet demand when that component fails, in the quantity set by `QUANTITY`. Plants are not drawn, since they sit at substations; the table above gives their unmet demand. A line's failure leaves demand unmet only when it separates part of the network from enough synchronous generation or overloads the lines that remain.

In [ ]:
QUANTITY = "unmet_mwh_per_day"  # or "unmet_peak_mw"
# clip: None for the islands with transmission lines, or a (minx, miny, maxx, maxy) box in degrees
h.plot_criticality_map(summary, QUANTITY, clip=None);

## Distribution network without supply when one component fails

The map shows the failure of the component set by `ASSET`: the distribution lines in the areas that lose supply, coloured by the share of the area's peak demand that is unmet, and the failed component in red. The box at the top left gives the two quantities and the causes of the unmet demand at peak.

In [ ]:
# asset_id of any component in summary; the table above lists those whose failure leaves demand unmet.
# The default is the component whose failure cuts off the most demand beyond its own area.
# Change iloc to select a different component based on its rank in cut_off_peak_mw.
ASSET = unmet.sort_values("cut_off_peak_mw", ascending=False, kind="stable")["asset_id"].iloc[0]
# clip: None for the component's island, or a (minx, miny, maxx, maxy) box in degrees
h.plot_failure(ASSET, summary=summary, by_area=by_area, clip=None);